In [1]:
import numpy
print(numpy.__version__)

2.5.1


## Projeto: "Convergência Estatística Aplicada à Precificação de Opções via Modelo Binomial"
Demonstrar empiricamente, através de simulação, como a LFGN, a LGGN e o TLC (via Teorema de De Moivre-Laplace) sustentam o método de precificação de opções por árvore binomial — e quantificar a incerteza da estimativa via Monte Carlo.

In [26]:
import numpy as np
import matplotlib.pyplot as mt
import scipy.stats
import yfinance as yf
import pandas as pd
import requests as rt
import math as m 

Obtenção dos dados que seram usados

In [27]:
dados = yf.download(
    "PETR4.SA",
    period="2y"
)



[*********************100%***********************]  1 of 1 completed


## Modulo  0: preparação e coleta dos dados

Calcular o Log-retorno diario

In [28]:
close  = dados['Close'].squeeze()
razao = close / close.shift(1) # razao entre o fechamento de hoje com o  dia anterior


log = np.log(razao) # calcula o log-retorno diario
log_retorno = log.dropna() 

dp = log_retorno.std() # desvio padrao
dp_anual = dp * np.sqrt(252) # volatilidade anual





print(dp_anual)



0.2461561114533046


Puxando API do banco central

In [29]:
codigo = 11
n = 1
url = f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados/ultimos/{n}?formato=json'
resposta = rt.get(url)
selic = resposta.json()
r = float(selic[0]['valor']) # Taxa livre de risco



Extraindo os restante dos dados

In [30]:
s0 = close.iat[-1] # Preço Atual da ação
strike = round(s0) # Preço em exercicio, arredondado
T = 0.25 # tempo de vencimento do mercado
n_steps = 200 # Divide em quantos remos vai ter ate a data de vencimento


OUTPUTS

In [31]:

def outputs(vol):
    deltaT = T/n_steps
    u = m.exp(vol * m.sqrt(deltaT))
    d = 1/u
    p = (m.exp(r * deltaT) - d) / (u - d)

    return u,d, p

u, d, p = outputs(dp_anual)

print(u, d, p)


1.0087409134024354 0.9913348281146318 0.5015968637932126


Função  simuladora de trajetoria

In [32]:
# Achando o valor de k 

def simulacao(s, P, sub, des, si, K_strike):
    k = np.random.binomial(s, P)
    Sn = si * m.pow(sub, k) * m.pow(des, (s - k))
    payoff = max(Sn - K_strike, 0)

    return Sn, k, payoff

Sn, k, payoff = simulacao(n_steps, p, u, d, s0, strike)

print(Sn, k, payoff)


45.74752623855103 103 2.747526238551032


## Modulo 1: Lei Forte dos Grandes Numeros